In [2]:
#imports
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
import random
layers = tf.keras.layers
Model = tf.keras.Model

In [3]:

print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

2.19.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [4]:
print(tf.config.experimental.list_physical_devices('GPU'))
import subprocess
print(subprocess.run(['nvidia-smi', 'topo', '-m'], capture_output=True, text=True).stdout)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
	GPU0	GPU1	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	PHB	0-3	0		N/A
GPU1	PHB	 X 	0-3	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks



In [5]:
strategy = tf.distribute.MirroredStrategy(cross_device_ops=tf.distribute.HierarchicalCopyAllReduce())
print(f"Number of devices: {strategy.num_replicas_in_sync}")

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Number of devices: 2


I0000 00:00:1782836436.251665    1851 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782836436.257816    1851 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [6]:
def load_character_images(root_dir):
    character_to_images = {}

    for alphabet in os.listdir(root_dir):
        alphabet_path = os.path.join(root_dir, alphabet)
        if not os.path.isdir(alphabet_path):
            continue

        for character in os.listdir(alphabet_path):
            character_path = os.path.join(alphabet_path, character)
            if not os.path.isdir(character_path):
                continue

            full_paths = [
                os.path.join(character_path, f)
                for f in os.listdir(character_path)
            ]

            key = f"{alphabet}_{character}"
            character_to_images[key] = full_paths

    return character_to_images


def proc_img(path1, path2, label):
    img1 = tf.io.read_file(path1)
    img2 = tf.io.read_file(path2)

    img1 = tf.image.decode_png(img1, channels=3)
    img2 = tf.image.decode_png(img2, channels=3)

    img1 = tf.image.resize(img1, (224, 224))
    img2 = tf.image.resize(img2, (224, 224))

    img1 = tf.cast(img1, tf.float32) / 255.0
    img2 = tf.cast(img2, tf.float32) / 255.0

    return (img1, img2), label


def create_dataset(pairs, batch_size=8):
    img1_paths = [p[0] for p in pairs]
    img2_paths = [p[1] for p in pairs]
    labels     = [p[2] for p in pairs]

    dataset = tf.data.Dataset.from_tensor_slices(
        ((img1_paths, img2_paths), labels)
    )

    options = tf.data.Options()
    options.experimental_deterministic = False
    dataset = dataset.with_options(options)

    dataset = dataset.map(
        lambda paths, label: proc_img(paths[0], paths[1], label),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # 1. Batch first (remember drop_remainder=True for multi-GPU)
    dataset = dataset.batch(batch_size, drop_remainder=True)

    # 2. Cache the batches in RAM
    dataset = dataset.cache()

    # 3. Shuffle (buffer size can be smaller now since it's shuffling batches)
    dataset = dataset.shuffle(
        buffer_size=100, 
        reshuffle_each_iteration=True
    )

    # 4. Prefetch for the GPU
    dataset = dataset.prefetch(tf.data.AUTOTUNE)


    return dataset

In [7]:
import random

def make_pairs(char_dict, num_pairs=20):
    class_names = list(char_dict.keys())
    all_pairs = []
    for current_class in class_names:
        images = char_dict[current_class]
        # same pairs (label = 1)
        used_same = set()
        while len(used_same) < num_pairs:
            img1 = random.choice(images)
            img2 = random.choice(images)
            while img2 == img1:
                img2 = random.choice(images)
            pair_check = tuple(sorted([img1, img2]))
            if pair_check in used_same:
                continue
            used_same.add(pair_check)
            all_pairs.append([img1, img2, 1])
        # different pairs (label = 0)
        used_diff = set()
        while len(used_diff) < num_pairs:
            class2 = random.choice([x for x in class_names if x != current_class])
            img1 = random.choice(images)
            img2 = random.choice(char_dict[class2])
            pair_check = tuple(sorted([img1, img2]))
            if pair_check in used_diff:
                continue
            used_diff.add(pair_check)
            all_pairs.append([img1, img2, 0])
    random.shuffle(all_pairs)
    return all_pairs

In [8]:
layers = tf.keras.layers
Model = tf.keras.Model

def build_encoder(input_shape=(224, 224, 3), embedding_dim=128, freeze_backbone=False):
    inputs = layers.Input(shape=input_shape, name="image")
    base_model = tf.keras.applications.ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape,
        pooling='avg'
    )
    base_model.trainable = not freeze_backbone
    x = base_model(inputs)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dense(embedding_dim)(x)
    embeddings = layers.Lambda(lambda t: tf.math.l2_normalize(t, axis=1))(x)
    return Model(inputs, embeddings)

def build_siamese_model(enc):
    img1 = layers.Input(shape=(224, 224, 3), name='img1')
    img2 = layers.Input(shape=(224, 224, 3), name='img2')
    tensor1 = enc(img1)
    tensor2 = enc(img2)
    distance = layers.Lambda(lambda tensors: tf.norm(tensors[0] - tensors[1], axis=1))([tensor1, tensor2])
    return Model([img1, img2], distance)

In [ ]:
char_dict = load_character_images('/kaggle/input/datasets/qweenink/omniglot/images_background_small1/images_background_small1')
pairs = make_pairs(char_dict)
dataset = create_dataset(pairs, batch_size=64)

dataset = strategy.experimental_distribute_dataset(dataset)

with strategy.scope():
    enc = build_encoder(freeze_backbone=False)
    siamese = build_siamese_model(enc)
    optimizer = tf.keras.optimizers.Adam(1e-5)

print(f"Total pairs: {len(pairs)}")
print(f"Sample pair: {pairs[0]}")

In [14]:
def contrastive_loss(labels, distances, margin=1.0):
    labels = tf.cast(labels, tf.float32)
    same = labels * tf.square(distances)
    diff = (1 - labels) * tf.square(tf.maximum(margin - distances, 0))
    return same + diff

@tf.function
def train_step(batch):
    (img1, img2), labels = batch

    with tf.GradientTape() as tape:
        distances = siamese([img1, img2], training=True)

        loss = contrastive_loss(labels, distances, margin=1.0)

        loss = tf.nn.compute_average_loss(
            loss,
            global_batch_size=GLOBAL_BATCH_SIZE
        )

    grads = tape.gradient(loss, siamese.trainable_variables)
    optimizer.apply_gradients(zip(grads, siamese.trainable_variables))

    return loss


@tf.function
def distributed_train_step(batch):
    per_replica_losses = strategy.run(train_step, args=(batch,))

    return strategy.reduce(
        tf.distribute.ReduceOp.SUM,
        per_replica_losses,
        axis=None
    )


In [ ]:
epochs = 10
epoch_losses = []

best_loss = float("inf")
patience = 3
no_improve = 0

GLOBAL_BATCH_SIZE = 64   # change if you changed batch_size



for epoch in range(epochs):
    batch_losses = []

    for batch in dataset:
        loss = distributed_train_step(batch)
        batch_losses.append(loss.numpy())

    epoch_loss = np.mean(batch_losses)
    epoch_losses.append(epoch_loss)

    print(f"Epoch {epoch + 1}, Loss: {epoch_loss:.6f}")

    siamese.save(f"/kaggle/working/siamese_epoch{epoch+1}.keras")

    if epoch_loss < best_loss - 1e-5:
        best_loss = epoch_loss
        no_improve = 0
        enc.save("/kaggle/working/encoder_best.keras")
        print(f"  -> Best encoder saved (loss={best_loss:.6f})")
    else:
        no_improve += 1
        print(f"  -> No improvement ({no_improve}/{patience})")

        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break


enc.save("/kaggle/working/encoder_final.keras")

plt.figure(figsize=(6,4))
plt.plot(epoch_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Curve")
plt.grid(True)

plt.savefig(
    "/kaggle/working/training_curve.png",
    dpi=150,
    bbox_inches="tight"
)
plt.close()

print("Done")

In [ ]:
enc.save("/kaggle/working/encoder_final.keras")
plt.figure(figsize=(6,4))
plt.plot(epoch_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Curve")
plt.grid(True)

plt.savefig(
    "/kaggle/working/training_curve.png",
    dpi=150,
    bbox_inches="tight"
)
plt.close()

## cub200 retraining

In [ ]:
# Create a folder in your writable working directory
!mkdir -p /kaggle/working/cub200

# Extract the .tgz file from the input directory into the working directory
!tar -xzf /kaggle/input/datasets/hridayeshrana/cub-200-2011/CUB_200_2011.tgz -C /kaggle/working/cub200

In [9]:
import os

def load_cub_images(images_dir):
    """
    Reads the CUB-200 folder structure directly from the images directory.
    Expected structure: images_dir/001.Black_footed_Albatross/Black_Footed_Albatross_0001_796111.jpg
    """
    bird_to_images = {}

    for bird_species in os.listdir(images_dir):
        species_path = os.path.join(images_dir, bird_species)
        
        # Skip if it's not a directory
        if not os.path.isdir(species_path):
            continue

        # Get all image paths for this specific bird
        full_paths = [
            os.path.join(species_path, f)
            for f in os.listdir(species_path)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
        
        if full_paths:
            bird_to_images[bird_species] = full_paths

    return bird_to_images

# --- HOW TO USE IT WITH YOUR EXISTING CODE ---

# 1. Point directly to the exact images folder you provided
cub_path = '/kaggle/working/cub200/CUB_200_2011/images' 
cub_dict = load_cub_images(cub_path)

print(f"Found {len(cub_dict)} bird species.")

# 2. Reuse your exact same make_pairs function!
# Making 40 pairs per bird since the dataset is bigger
cub_pairs = make_pairs(cub_dict, num_pairs=40) 

# 3. Reuse your exact same create_dataset function!
# (Make sure drop_remainder=True is still in your create_dataset function)
cub_dataset = create_dataset(cub_pairs, batch_size=16)

# 4. Distribute it across your two T4 GPUs
cub_dataset = strategy.experimental_distribute_dataset(cub_dataset)

print(f"Total CUB pairs ready for training: {len(cub_pairs)}")

Found 200 bird species.
Total CUB pairs ready for training: 16000


In [10]:
with strategy.scope():
    enc = build_encoder()

    enc.load_weights('/kaggle/working/encoder_final.keras')
    # Rebuild the Siamese wrapper around your pre-trained encoder
    siamese = build_siamese_model(enc)
    
    # Initial learning rate for fine-tuning
    initial_lr = 1e-5
    optimizer = tf.keras.optimizers.Adam(initial_lr) 

    enc.summary()
    siamese.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 2048)           │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 128)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,801,024 (94.61 MB)

 Trainable params: 24,747,904 (94.41 MB)

 Non-trainable params: 53,120 (207.50 KB)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ img1 (InputLayer)   │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ img2 (InputLayer)   │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional          │ (None, 128)       │ 24,801,024 │ img1[0][0],       │
│ (Functional)        │                   │            │ img2[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None)            │          0 │ functional[0][0], │
│                     │                   │            │ functional[1][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,801,024 (94.61 MB)

 Trainable params: 24,747,904 (94.41 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [ ]:
import numpy as np

# 1. Define the loss function
def contrastive_loss(labels, distances, margin=1.0):
    labels = tf.cast(labels, tf.float32)
    same = labels * tf.square(distances)
    diff = (1 - labels) * tf.square(tf.maximum(margin - distances, 0))
    return same + diff

# 2. Define the core training step (runs on a single device)
@tf.function
def train_step(batch):
    (img1, img2), labels = batch
    with tf.GradientTape() as tape:
        distances = siamese([img1, img2], training=True)
        loss = tf.reduce_mean(contrastive_loss(labels, distances))
        
    grads = tape.gradient(loss, siamese.trainable_variables)
    optimizer.apply_gradients(zip(grads, siamese.trainable_variables))
    return loss

# 3. Define the distributed wrapper (unwraps PerReplica data)
@tf.function
def distributed_train_step(batch):
    per_replica_losses = strategy.run(train_step, args=(batch,))
    return strategy.reduce(tf.distribute.ReduceOp.MEAN, per_replica_losses, axis=None)

# 4. Training loop
best_loss = float("inf")
patience_counter = 0

print("Starting Training...")
for epoch in range(10):
    batch_losses = []
    
    for batch in cub_dataset:
        loss = distributed_train_step(batch)
        batch_losses.append(loss.numpy())
    
    avg_loss = np.mean(batch_losses)
    print(f"Epoch {epoch + 1}, Loss: {avg_loss:.6f}")
    
    # Checkpointing and LR Reduction logic
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        siamese.save("/kaggle/working/best_model.keras")
        print("  -> Improvement! Saved best model.")
    else:
        patience_counter += 1
        print(f"  -> No improvement ({patience_counter}/3)")
        if patience_counter >= 3:
            new_lr = float(optimizer.learning_rate.numpy()) * 0.5
            optimizer.learning_rate.assign(new_lr)
            patience_counter = 0
            print(f"  -> Reducing LR to {new_lr:.2e}")

Starting Training...
INFO:tensorflow:batch_all_reduce: 218 all-reduces with algorithm = hierarchical_copy, num_packs = 1
INFO:tensorflow:batch_all_reduce: 218 all-reduces with algorithm = hierarchical_copy, num_packs = 1
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


I0000 00:00:1782835668.725760    1589 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1782835668.725763    1587 cuda_dnn.cc:529] Loaded cuDNN version 91002
